# RegionAnalyzer — synthetic 3D segmentation

Build a small labeled volume (three non-overlapping objects) and extract region properties as a pandas DataFrame.

In [ ]:
import numpy as np
import pandas as pd
import stackview

from vistiq.utils import ArrayIteratorConfig
from vistiq.segment.select import FULL, LOWER, UPPER, LOWER_ND, UPPER_ND, OFF_DIAGONAL
from vistiq.segment.analysis import RegionAnalyzer, RegionAnalyzerConfig, region_to_numpy, dataframe_to_numpy
from vistiq.segment.select import RegionFilter, RegionFilterConfig, MinFilterConfig, RangeFilterConfig
from vistiq.segment.select import ValueFilter, ValueFilterConfig, TopKFilter, TopKFilterConfig

## Synthetic label volume

Shape `(100, 100, 10)` with labels `1`, `2`, and `3` in separate spatial regions (no overlap), mimicking a 3D segmentation mask.

In [ ]:
labels = np.zeros((10, 200, 200), dtype=np.uint64)

# Object 1 — upper-left
labels[2:7, 20:38, 12:30] = 1

# Object 2 — center
labels[3:9, 42:58, 38:62] = 2

# Object 3 — lower-right
labels[1:6, 72:92, 68:88] = 3

# Object 4 — lower-right
labels[4:6, 112:132, 125:163] = 4

# Object 5 — lower-right
labels[7:9, 145:180, 12:58] = 5

unique_labels = np.unique(labels)
print(f"labels.shape={labels.shape}, dtype={labels.dtype}")
print(f"unique labels: {unique_labels}")
print(f"voxel counts: {{{', '.join(f'{int(l)}: {int((labels == l).sum())}' for l in unique_labels if l)}}}")

In [ ]:
areas = np.zeros((10, 200, 200), dtype=np.uint64)

# Area 1 — upper-left
areas[2:9, 10:58, 10:98] = 6


# Area2 — lower-right
areas[1:10, 96:192, 58:178] = 8

unique_labels = np.unique(labels)
print(f"labels.shape={labels.shape}, dtype={labels.dtype}")
print(f"unique labels: {unique_labels}")
print(f"voxel counts: {{{', '.join(f'{int(l)}: {int((labels == l).sum())}' for l in unique_labels if l)}}}")

In [ ]:
stackview.slice(np.concatenate([labels, areas], axis=-1))

## RegionAnalyzer (dataframe output)

Analyze the full 3D volume (`slice_def=()`). With `map_axes=True`, vector properties such as `cross_sectional_area` and `aspect_ratio` are expanded to plane-specific columns (`-xy`, `-xz`, `-yz`).

In [ ]:
metadata = {
    "axes": ["Z", "Y", "X"],
    "scale": (2.0, 1.0, 1.0),
}

config = RegionAnalyzerConfig(
    output_type="dataframe",
    map_axes=True,
    properties=[
        "label",
        "volume",
        "centroid",
        "bbox",
        "aspect_ratio",
        "cross_sectional_area",
    ],
    iterator_config=ArrayIteratorConfig(slice_def=()),
)

l_regions = RegionAnalyzer(config).run(labels, metadata=metadata)
a_regions = RegionAnalyzer(config).run(areas, metadata=metadata)

In [ ]:
l_regions

In [ ]:
a_regions

# Filter Regions

In [ ]:
rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="volume",
            range=(100.0,np.inf)
        ),
        MinFilterConfig(
            attribute="aspect_ratio",
            minimum=0.015,
        ),
    ]
)
l_accepted, _ = RegionFilter(rfcfg).run(l_regions)
a_accepted, _ = RegionFilter(rfcfg).run(a_regions)

In [ ]:
l_accepted

In [ ]:
a_accepted

# Calculate inter-object distances

Uses PyTorch tensors. The config allows setting a `preferred_device` ("cuda", "mps", "cpu", None). This is not a guarantee. The actual device can be assigned at runtime with `device`. If the device is None, it will be auto-discovered considering the config's `preferred_device`.

In [ ]:
dccfg = DistanceCalculatorConfig(
    annotate=True, 
    output_type="torch.Tensor",
    preferred_device="cuda",
)

centroids = dataframe_to_numpy(l_accepted, attributes=["centroid"], strict=False)
object_ids = dataframe_to_numpy(l_accepted, attributes=["object_id"])
dist = DistanceCalculator(dccfg).run(
    centroids, 
    centroids, 
    spacing=metadata.get("scale", None), 
    point_annotations=(object_ids, object_ids),
    device=None
)

In [ ]:
type(dist), dist

# Apply a Rank Filter

`axis=0`: column-wise
`axis=1`: row-wise
`axis=None`: global

`output` options: ["masked_values", "indices", "mask", "values"]

In [ ]:
tkcfg = TopKFilterConfig(
    k=1,
    axis=1,
    largest=False,
    triangle=OFF_DIAGONAL,
    output="masked_values",
)

tk = TopKFilter(tkcfg).run(dist)
tk

# Apply a Threshold based Filter

In [ ]:
mincfg = ValueFilterConfig(
    ref_value=80.0,
    axis=0,
    operator=">",
    triangle=LOWER_ND,
    output="masked_values",
)
maxcfg = ValueFilterConfig(
    ref_value=120.0,
    axis=0,
    operator="<",
    triangle=LOWER_ND,
    output="masked_values",
)
mint = ValueFilter(mincfg).run(dist)
mint

# Aggregate

In [ ]:
macfg = MatrixAggregatorConfig(
    operation="sum",
    axis=1,
)

counts = MatrixAggregator(macfg).run(ranget)
counts